In [21]:
import pyspark
from pyspark.sql import SparkSession

In [22]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [ ]:
df_green = spark.read.parquet('data/pq/green/*/*')

In [ ]:
df_green.registerTempTable('green')

In [28]:
df_result_green = spark.sql("""
SELECT
    -- Revenue grouping
    PULocationID AS revenue_zone,
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    SUM(total_amount) AS amount,
    COUNT(1) as number_records

FROM
    green
WHERE  lpep_pickup_datetime > '2020-01-01 00:00:00'
GROUP BY
    1, 2
ORDER BY
    1, 2
""")

In [ ]:
df_result_green.write.parquet('data/report/revenue/green', mode='overwrite')

In [ ]:
df_yellow = spark.read.parquet('data/pq/yellow/*/*')
df_yellow.registerTempTable('yellow')
df_result_yellow = spark.sql("""
SELECT
    -- Revenue grouping
    PULocationID AS revenue_zone,
    date_trunc('hour', tpep_pickup_datetime) AS hour,
    SUM(total_amount) AS amount,
    COUNT(1) as number_records

FROM
    yellow
WHERE  tpep_pickup_datetime > '2020-01-01 00:00:00'
GROUP BY
    1, 2
ORDER BY
    1, 2
""")

In [35]:
df_green_rev_temp = df_result_green\
    .withColumnRenamed('number_records', 'green_number_records')\
    .withColumnRenamed('amount', 'green_amount')

df_yellow_rev_temp = df_result_yellow\
    .withColumnRenamed('number_records', 'yellow_number_records')\
    .withColumnRenamed('amount', 'yellow_amount')

In [36]:
df_join = df_green_rev_temp.join(df_yellow_rev_temp, on=['hour', 'revenue_zone'], how='outer')

In [38]:
df_join.write.parquet('data/report/revenue/total', mode='overwrite')

26/11/21 18:14:06 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/11/21 18:14:06 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/11/21 18:14:06 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/11/21 18:14:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/11/21 18:14:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
